# Manual RAG Pipeline: Mechanisms First

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline from scratch.
You'll see every step explicitly before we move to frameworks like LangChain.

**Works on:** Google Colab, Local Jupyter (Mac/Windows/Linux)

**Pipeline Overview:**
```
Documents → Chunking → Embedding → Index (FAISS)
                                        ↓
User Query → Embed Query → Similarity Search → Top-K Chunks
                                                    ↓
                                        Prompt Assembly → LLM → Answer
```

## TODO — Topic 5 RAG Course Project Checklist

- **Exercise 0:** Set-up — Get notebook running; unzip Corpora.zip. Use PDFs from `Corpora/<corpus>/pdf_embedded/`.
- **Exercise 1:** Open model RAG vs no RAG — Compare Qwen 2.5 1.5B with/without RAG on Model T manual and Congressional Record.
- **Exercise 2:** Open model + RAG vs large model — Run GPT-4o Mini with no tools on same queries.
- **Exercise 3:** Open model + RAG vs frontier chat — Compare local Qwen+RAG vs GPT-4/Claude (web).
- **Exercise 4:** Effect of top-K — Test k = 1, 3, 5, 10, 20.
- **Exercise 5:** Unanswerable questions — Off-topic, related-but-missing, false premise.
- **Exercise 6:** Query phrasing sensitivity — Same question in 5+ phrasings.
- **Exercise 7:** Chunk overlap — Re-chunk with overlap 0, 64, 128, 256.
- **Exercise 8:** Chunk size — Chunk at 128, 256, 512, 1024, 2048.
- **Exercise 9:** Retrieval score analysis — 10 queries, top-10 chunks, score distribution.
- **Exercise 10:** Prompt template variations — Minimal, strict grounding, citation, permissive, structured.
- **Exercise 11:** Failure mode catalog — Computation, temporal, comparison, ambiguous, multi-hop, etc.
- **Exercise 12:** Cross-document synthesis — Questions needing multiple chunks.

## Setup

First, let's install the required packages and detect our compute environment.

In [1]:
# Install dependencies
# On Colab, these install quickly. Locally, you may already have them.
# Use a kernel-aware install when available; fall back to subprocess otherwise.
try:
    ip = get_ipython()
    ip.run_line_magic('pip', 'install -q torch transformers sentence-transformers faiss-cpu pymupdf accelerate ipyfilechooser')
except NameError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'transformers', 'sentence-transformers', 'faiss-cpu', 'pymupdf', 'accelerate', 'ipyfilechooser'])
# For Exercise 2 (GPT-4o Mini): add 'openai' to the list above if needed


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 109.4 MB/s eta 0:00:00


In [2]:
# =============================================================================
# ENVIRONMENT AND DEVICE DETECTION
# =============================================================================
import os
import sys

# Enable MPS fallback for any PyTorch operations not yet implemented on Metal
# This MUST be set before importing torch
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# Prevent kernel crash from duplicate OpenMP libraries (PyTorch + FAISS conflict on macOS)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
from typing import Tuple

def detect_environment() -> str:
    """Detect if we're running on Colab or locally."""
    try:
        import google.colab
        return 'colab'
    except ImportError:
        return 'local'

def get_device() -> Tuple[str, torch.dtype]:
    """
    Detect the best available compute device.

    Priority: CUDA > MPS (Apple Silicon) > CPU

    Returns:
        Tuple of (device_string, recommended_dtype)

    Notes:
        - CUDA: Use float16 for memory efficiency (Tensor Cores optimize this)
        - MPS: Use float32 - Apple Silicon doesn't have the same float16
               optimizations as NVIDIA, and float32 is often faster
        - CPU: Use float32 (float16 not well supported on CPU)
    """
    if torch.cuda.is_available():
        device = 'cuda'
        dtype = torch.float16
        device_name = torch.cuda.get_device_name(0)
        memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✓ Using CUDA GPU: {device_name} ({memory_gb:.1f} GB)")

    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        device = 'mps'
        dtype = torch.float32  # float32 is often faster on Apple Silicon!
        print("✓ Using Apple Silicon GPU (MPS)")
        print("  Note: Using float32 (faster than float16 on Apple Silicon)")

    else:
        device = 'cpu'
        dtype = torch.float32
        print("⚠ Using CPU (no GPU detected)")
        print("  Tip: For faster processing, use a machine with a GPU")

    return device, dtype

# Detect environment and device
ENVIRONMENT = detect_environment()
DEVICE, DTYPE = get_device()

print(f"\nEnvironment: {ENVIRONMENT.upper()}")
print(f"Device: {DEVICE}, Dtype: {DTYPE}")

✓ Using CUDA GPU: NVIDIA H100 80GB HBM3 (85.0 GB)

Environment: COLAB
Device: cuda, Dtype: torch.float16


## Load Your Documents

**Cell 1:** Configure your document source and select/upload files
- **Local Jupyter**: Use the folder picker, then run Cell 2
- **Colab + Upload**: Files upload immediately (blocking), then run Cell 2
- **Colab + Drive**: Set `USE_GOOGLE_DRIVE = True`, mounts Drive and shows picker, then run Cell 2

**Cell 2:** Confirms selection and lists documents

In [19]:
# =============================================================================
# CELL 1: SELECT DOCUMENT SOURCE
# =============================================================================
# This cell either:
#   - Shows a folder picker (Local or Colab+Drive) - NON-BLOCKING
#   - Shows an upload dialog (Colab+Upload) - BLOCKING
#
# If a folder picker is shown, SELECT YOUR FOLDER BEFORE running Cell 2.
# The picker widget is non-blocking, so the code continues before you select.
# =============================================================================

from pathlib import Path

# ------------- COLAB USERS: CONFIGURE HERE -------------
USE_GOOGLE_DRIVE = True  # Set to True to use Google Drive instead of uploading
# -------------------------------------------------------

# Default folder: use Corpora from course project (unzip Corpora.zip first).
_folder_default = Path("Corpora/ModelTService")
DOC_FOLDER = str(_folder_default) if _folder_default.exists() else "documents"
folder_chooser = None  # Will hold the picker widget if used

if ENVIRONMENT == 'colab':
    if USE_GOOGLE_DRIVE:
        # ----- COLAB + GOOGLE DRIVE -----
        # Mount Drive first, then show folder picker
        from google.colab import drive
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
        print("✓ Google Drive mounted\n")

        # Now show folder picker for the Drive
        try:
            from ipyfilechooser import FileChooser

            folder_chooser = FileChooser(
                path='/content/drive/MyDrive',
                title='Select your documents folder in Google Drive',
                show_only_dirs=True,
                select_default=True
            )
            print("📁 Select your documents folder below, then run Cell 2:")
            print("   (The picker is non-blocking - select BEFORE running the next cell)")
            display(folder_chooser)

        except ImportError:
            # Fallback: manual path entry
            print("Folder picker not available.")
            print("Edit DOC_FOLDER below with your Google Drive path, then run Cell 2:")
            DOC_FOLDER = '/content/drive/MyDrive/your_documents_folder'  # ← Edit this!
            print(f"  DOC_FOLDER = '{DOC_FOLDER}'")
    else:
        # ----- COLAB + UPLOAD -----
        # Upload dialog blocks until complete, so DOC_FOLDER is ready when done
        from google.colab import files
        os.makedirs(DOC_FOLDER, exist_ok=True)

        print("Upload your documents (PDF, TXT, or MD):")
        print("(This dialog blocks until upload is complete)\n")
        uploaded = files.upload()

        for filename in uploaded.keys():
            os.rename(filename, f'{DOC_FOLDER}/{filename}')
            print(f"  ✓ Saved: {DOC_FOLDER}/{filename}")

        print(f"\n✓ Upload complete. Run Cell 2 to continue.")

else:
    # ----- LOCAL JUPYTER -----
    # Show folder picker
    print("Running locally\n")

    try:
        from ipyfilechooser import FileChooser

        folder_chooser = FileChooser(
            path=str(Path.home()),
            title='Select your documents folder',
            show_only_dirs=True,
            select_default=True
        )
        print("📁 Select your documents folder below, then run Cell 2:")
        print("   (The picker is non-blocking - select BEFORE running the next cell)")
        display(folder_chooser)

    except ImportError:
        # Fallback: manual path entry
        print("Folder picker not available (ipyfilechooser not installed).")
        print(f"\nUsing default folder: {Path(DOC_FOLDER).absolute()}")
        print("\nTo use a different folder, edit DOC_FOLDER in this cell:")
        print("  DOC_FOLDER = '/path/to/your/documents'")
        os.makedirs(DOC_FOLDER, exist_ok=True)

Mounting Google Drive...
Mounted at /content/drive
✓ Google Drive mounted

📁 Select your documents folder below, then run Cell 2:
   (The picker is non-blocking - select BEFORE running the next cell)


FileChooser(path='/content/drive/MyDrive', filename='', title='Select your documents folder in Google Drive', …

In [ ]:
# =============================================================================
# CELL 2: CONFIRM SELECTION AND LIST DOCUMENTS
# =============================================================================
# If you used a folder picker above, make sure you selected a folder
# BEFORE running this cell. The picker is non-blocking.
# =============================================================================

# Read selection from folder picker (if one was used)
if folder_chooser is not None and folder_chooser.selected_path:
    DOC_FOLDER = folder_chooser.selected_path
    print(f"✓ Using selected folder: {DOC_FOLDER}")
elif folder_chooser is not None:
    print("⚠ No folder selected in picker!")
    print("  Please go back to Cell 1, select a folder, then run this cell again.")
else:
    # No picker used (upload or manual path)
    print(f"✓ Using folder: {DOC_FOLDER}")

# Confirm folder (listing skipped for speed)
doc_path = Path(DOC_FOLDER)
if doc_path.exists():
    print(f"✓ Folder set: {doc_path.absolute()}")
    print("  Run the next cells to load, chunk, and index documents.")
else:
    print(f"⚠ Folder not found: {DOC_FOLDER}")
    print("  Please set DOC_FOLDER in the previous cell and run it again.")

✓ Using selected folder: /content/drive/MyDrive/Corpora/Congressional_Record_Jan_2026/pdf_embedded
✓ Folder set: /content/drive/MyDrive/Corpora/Congressional_Record_Jan_2026/pdf_embedded
  Run the next cells to load, chunk, and index documents.


---
## Stage 1: Document Loading

We need to extract text from our documents. For PDFs with embedded text,
PyMuPDF (fitz) reads the text layer directly - no OCR needed.

**Corpora:** Use PDFs from `Corpora/<name>/pdf_embedded/`. The `.txt` files in `txt/` are for checking retrieval vs OCR issues.

In [ ]:
# Exercise 1 (and reuse): Official query lists. Reference: CR Jan 13, 20, 21, 23, 2026.
QUERIES_MODEL_T = [
    "How do I adjust the carburetor on a Model T?",
    "What is the correct spark plug gap for a Model T Ford?",
    "How do I fix a slipping transmission band?",
    "What oil should I use in a Model T engine?",
]
QUERIES_CR = [
    "What did Mr. Flood have to say about Mayor David Black in Congress on January 13, 2026?",
    "What mistake did Elise Stefanik make in Congress on January 23, 2026?",
    "What is the purpose of the Main Street Parity Act?",
    "Who in Congress has spoken for and against funding of pregnancy centers?",
]

In [17]:
import fitz  # PyMuPDF
from typing import List, Tuple

def load_text_file(filepath: str) -> str:
    """Load a plain text file."""
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()


def load_pdf_file(filepath: str) -> str:
    """
    Extract text from a PDF with embedded text.

    PyMuPDF reads the text layer directly.
    For scanned PDFs without embedded text, you'd need OCR.
    """
    doc = fitz.open(filepath)
    text_parts = []

    for page_num, page in enumerate(doc):
        text = page.get_text()
        if text.strip():
            # Add page marker for debugging/citation
            text_parts.append(f"\n[Page {page_num + 1}]\n{text}")

    doc.close()
    return "\n".join(text_parts)


def load_documents(doc_folder: str) -> List[Tuple[str, str]]:
    """Load all documents from a folder. Returns list of (filename, content)."""
    documents = []
    folder = Path(doc_folder)

    for filepath in folder.rglob("*"):
        try:
            if not filepath.is_file():
                continue
        except OSError:
            continue
        if filepath.suffix.lower() not in ('.pdf', '.txt', '.md', '.text'):
            continue
        try:
            if filepath.suffix.lower() == '.pdf':
                content = load_pdf_file(str(filepath))
            elif filepath.suffix.lower() in ['.txt', '.md', '.text']:
                content = load_text_file(str(filepath))
            else:
                continue

            if content.strip():
                documents.append((filepath.name, content))
                print(f"✓ Loaded: {filepath.name} ({len(content):,} chars)")
        except Exception as e:
            print(f"✗ Error loading {filepath}: {e}")

    return documents

In [ ]:
# Load your documents
documents = load_documents(DOC_FOLDER)
print(f"\nLoaded {len(documents)} documents")

if len(documents) == 0:
    print("\n⚠ No documents loaded! Please add PDF or TXT files to the documents folder.")

✓ Loaded: CREC-2026-01-03-v171.pdf (21,119 chars)
✓ Loaded: CREC-2026-01-29.pdf (428,669 chars)
✓ Loaded: CREC-2026-01-16.pdf (76,821 chars)
✓ Loaded: CREC-2026-01-05.pdf (161,797 chars)
✓ Loaded: CREC-2026-01-06.pdf (1,021,965 chars)
✓ Loaded: CREC-2026-01-07.pdf (707,729 chars)
✓ Loaded: CREC-2026-01-08.pdf (1,357,469 chars)
✓ Loaded: CREC-2026-01-08-bk2.pdf (29,597 chars)
✓ Loaded: CREC-2026-01-08-bk3.pdf (1,176,080 chars)
✓ Loaded: CREC-2026-01-09.pdf (234,300 chars)
✓ Loaded: CREC-2026-01-12.pdf (647,478 chars)
✓ Loaded: CREC-2026-01-20.pdf (1,128,096 chars)
✓ Loaded: CREC-2026-01-21.pdf (464,955 chars)
✓ Loaded: CREC-2026-01-22.pdf (1,886,465 chars)
✓ Loaded: CREC-2026-01-22-bk2.pdf (1,659,911 chars)
✓ Loaded: CREC-2026-01-02.pdf (18,465 chars)
✓ Loaded: CREC-2026-01-13.pdf (987,163 chars)
✓ Loaded: CREC-2026-01-14.pdf (1,743,893 chars)
✓ Loaded: CREC-2026-01-15.pdf (626,449 chars)
✓ Loaded: CREC-2026-01-23.pdf (19,750 chars)
✓ Loaded: CREC-2026-01-26.pdf (20,719 chars)
✓ Loaded:

In [ ]:
# Inspect a document to verify loading worked
if documents:
    filename, content = documents[0]
    print(f"First document: {filename}")
    print(f"Total length: {len(content):,} characters")
    print(f"\nFirst 1000 characters:\n{'-'*40}")
    print(content[:1000])

First document: CREC-2026-01-03-v171.pdf
Total length: 21,119 characters

First 1000 characters:
----------------------------------------

[Page 1]
Congressional Record
U
N
U
M
E
P
LU
RI
B
U
S
United States
of America
PROCEEDINGS AND DEBATES OF THE 119th CONGRESS, FIRST SESSION
b This symbol represents the time of day during the House proceedings, e.g., b 1407 is 2:07 p.m.
Matter set in this typeface indicates words inserted or appended, rather than spoken, by a Member of the House on the floor.
.
H6137 
Vol. 171 
WASHINGTON, SATURDAY, JANUARY 3, 2026 
No. 220 
Senate 
The Senate was not in session today. Its next meeting will be held on Saturday, January 3, 2026, at 12 p.m. 
House of Representatives 
SATURDAY, JANUARY 3, 2026 
The House met at 11:30 a.m. and was 
called to order by the Speaker pro tem-
pore (Mr. SMITH of New Jersey). 
f 
DESIGNATION OF THE SPEAKER 
PRO TEMPORE 
The SPEAKER pro tempore laid be-
fore the House the following commu-
nication from the Speaker: 
WASHINGTON,

---
## Stage 2: Chunking

Documents need to be split into pieces small enough to be relevant but large enough to carry meaning.

**Why overlap?** If a key sentence sits right at a chunk boundary, splitting without overlap might cut it in half. Overlap ensures that information near boundaries appears intact in at least one chunk.

**Experiment:** Try different chunk sizes (256, 512, 1024) and see how it affects retrieval!

In [20]:
from dataclasses import dataclass

@dataclass
class Chunk:
    """A chunk of text with metadata for tracing back to source."""
    text: str
    source_file: str
    chunk_index: int
    start_char: int
    end_char: int


def chunk_text(
    text: str,
    source_file: str,
    chunk_size: int = 512,
    chunk_overlap: int = 128
) -> List[Chunk]:
    """
    Split text into overlapping chunks.

    We try to break at sentence or paragraph boundaries
    to avoid cutting mid-thought.
    """
    chunks = []
    start = 0
    chunk_index = 0

    while start < len(text):
        end = start + chunk_size

        # Try to break at a good boundary
        if end < len(text):
            # Look for paragraph break first
            para_break = text.rfind('\n\n', start + chunk_size // 2, end)
            if para_break != -1:
                end = para_break + 2
            else:
                # Look for sentence break
                sentence_break = text.rfind('. ', start + chunk_size // 2, end)
                if sentence_break != -1:
                    end = sentence_break + 2

        chunk_text_str = text[start:end].strip()

        if chunk_text_str:
            chunks.append(Chunk(
                text=chunk_text_str,
                source_file=source_file,
                chunk_index=chunk_index,
                start_char=start,
                end_char=end
            ))
            chunk_index += 1

        # Move forward, accounting for overlap
        start = end - chunk_overlap
        if chunks and start <= chunks[-1].start_char:
            start = end  # Safety: ensure progress

    return chunks

In [ ]:
# ============================================
# EXPERIMENT: Try different chunk sizes!
# ============================================
CHUNK_SIZE = 512      # Try: 256, 512, 1024
CHUNK_OVERLAP = 128   # Try: 64, 128, 256
# For Ex 7/8 use rebuild_pipeline() — see cell after FAISS index.

# Chunk all documents
all_chunks = []
for filename, content in documents:
    doc_chunks = chunk_text(content, filename, CHUNK_SIZE, CHUNK_OVERLAP)
    all_chunks.extend(doc_chunks)
    print(f"{filename}: {len(doc_chunks)} chunks")

print(f"\nTotal: {len(all_chunks)} chunks")

CREC-2026-01-03-v171.pdf: 65 chunks
CREC-2026-01-29.pdf: 1323 chunks
CREC-2026-01-16.pdf: 254 chunks
CREC-2026-01-05.pdf: 494 chunks
CREC-2026-01-06.pdf: 3044 chunks
CREC-2026-01-07.pdf: 2280 chunks
CREC-2026-01-08.pdf: 4206 chunks
CREC-2026-01-08-bk2.pdf: 92 chunks
CREC-2026-01-08-bk3.pdf: 3597 chunks
CREC-2026-01-09.pdf: 754 chunks
CREC-2026-01-12.pdf: 2062 chunks
CREC-2026-01-20.pdf: 3185 chunks
CREC-2026-01-21.pdf: 1483 chunks
CREC-2026-01-22.pdf: 5688 chunks
CREC-2026-01-22-bk2.pdf: 4971 chunks
CREC-2026-01-02.pdf: 62 chunks
CREC-2026-01-13.pdf: 3120 chunks
CREC-2026-01-14.pdf: 5475 chunks
CREC-2026-01-15.pdf: 1981 chunks
CREC-2026-01-23.pdf: 61 chunks
CREC-2026-01-26.pdf: 67 chunks
CREC-2026-01-26-bk2.pdf: 44 chunks
CREC-2026-01-27.pdf: 779 chunks
CREC-2026-01-28.pdf: 1695 chunks
CREC-2026-01-30.pdf: 1096 chunks

Total: 47878 chunks


In [ ]:
# Inspect some chunks
if all_chunks:
    print("Sample chunks:")
    indices_to_show = [0, len(all_chunks)//2, -1] if len(all_chunks) > 2 else range(len(all_chunks))
    for i in indices_to_show:
        chunk = all_chunks[i]
        print(f"\n{'='*60}")
        print(f"Chunk {chunk.chunk_index} from {chunk.source_file}")
        print(f"{'='*60}")
        print(chunk.text[:300] + "..." if len(chunk.text) > 300 else chunk.text)

Sample chunks:

Chunk 0 from CREC-2026-01-03-v171.pdf
[Page 1]
Congressional Record
U
N
U
M
E
P
LU
RI
B
U
S
United States
of America
PROCEEDINGS AND DEBATES OF THE 119th CONGRESS, FIRST SESSION
b This symbol represents the time of day during the House proceedings, e.g., b 1407 is 2:07 p.m.
Matter set in this typeface indicates words inserted or appende...

Chunk 1100 from CREC-2026-01-22.pdf
ion Act of 
2011, sections 405(a) and 406 of the Trade 
Preferences Extension Act of 2015, and sec-
tion 285(a) of the Trade Act of 1974, as 
amended, and shall be available for Federal 
obligation through December 31, 2026, except 
that funds for outcome payments pursuant 
to section 306(f)(2) of t...

Chunk 1095 from CREC-2026-01-30.pdf
mmy, Fla., E81, E85 
Stauber, Pete, Minn., E81 
Zinke, Ryan K., Mont., E84 
VerDate Sep 11 2014 
06:48 Jan 31, 2026
Jkt 069060
PO 00000
Frm 00004
Fmt 0664
Sfmt 0664
E:\CR\FM\D30JA6.REC
D30JAPT1
dmwilson on DSK7X7S144PROD with DIGEST


---
## Stage 3: Embedding

Embeddings map text to dense vectors where **semantic similarity = geometric proximity**.

A sentence about "cardiac arrest" and one about "heart attack" will have similar embeddings even though they share no words.

**Note:** sentence-transformers does NOT auto-detect Apple MPS - we must pass the device explicitly.

In [21]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
# Options:
# - "sentence-transformers/all-MiniLM-L6-v2": Fast, small (80MB), good quality
# - "BAAI/bge-small-en-v1.5": Better for retrieval, similar size

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {EMBEDDING_MODEL}")
print(f"Device: {DEVICE}")

# Must explicitly pass device for MPS support!
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
EMBEDDING_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {EMBEDDING_DIM}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
Device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


In [22]:
# DEMO: See how embeddings capture semantic similarity
test_sentences = [
    "The engine needs regular oil changes.",
    "Motor oil should be replaced periodically.",
    "The Senate convened at noon.",
    "Congress began its session at midday."
]

test_embeddings = embed_model.encode(test_sentences)

# Compute cosine similarity matrix
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print("Cosine similarity matrix:")
print("\n" + " " * 40 + "  [0]    [1]    [2]    [3]")
for i, s1 in enumerate(test_sentences):
    sims = [cosine_sim(test_embeddings[i], test_embeddings[j]) for j in range(4)]
    print(f"[{i}] {s1[:35]:35} {sims[0]:.3f}  {sims[1]:.3f}  {sims[2]:.3f}  {sims[3]:.3f}")

print("\n→ Notice: [0]-[1] are similar (both about oil), [2]-[3] are similar (both about Congress)")

Cosine similarity matrix:

                                          [0]    [1]    [2]    [3]
[0] The engine needs regular oil change 1.000  0.728  -0.045  -0.032
[1] Motor oil should be replaced period 0.728  1.000  0.014  0.035
[2] The Senate convened at noon.        -0.045  0.014  1.000  0.684
[3] Congress began its session at midda -0.032  0.035  0.684  1.000

→ Notice: [0]-[1] are similar (both about oil), [2]-[3] are similar (both about Congress)


In [ ]:
# Embed all chunks - this may take a few minutes for large corpora
if all_chunks:
    print(f"Embedding {len(all_chunks)} chunks on {DEVICE}...")
    chunk_texts = [c.text for c in all_chunks]
    chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=True)
    chunk_embeddings = chunk_embeddings.astype('float32')  # FAISS wants float32
    print(f"Embeddings shape: {chunk_embeddings.shape}")
else:
    print("No chunks to embed - please load documents first.")

Embedding 47878 chunks on cuda...


Batches:   0%|          | 0/1497 [00:00<?, ?it/s]

Embeddings shape: (47878, 384)


---
## Stage 4: Vector Index (FAISS)

FAISS efficiently finds nearest neighbors in high-dimensional spaces.

We use a simple **flat index** (brute-force search) which is transparent and works well for up to ~100k vectors. For larger corpora, you'd use approximate methods like IVF or HNSW.

**Note:** FAISS GPU support is CUDA-only. On MPS/CPU, we use faiss-cpu (still very fast for <100k vectors).

In [14]:
import faiss

In [ ]:
# Create FAISS index
# IndexFlatIP = Inner Product (for cosine similarity on normalized vectors)
index = faiss.IndexFlatIP(EMBEDDING_DIM)

if all_chunks:
    # Normalize vectors so inner product = cosine similarity
    faiss.normalize_L2(chunk_embeddings)

    # Add vectors to index
    index.add(chunk_embeddings)
    print(f"Index built with {index.ntotal} vectors")
else:
    print("No embeddings to index - please load and embed documents first.")

Index built with 47878 vectors


---
## Stage 5: Retrieval

Now we can search! Given a query, we:
1. Embed the query with the same model
2. Find the top-k most similar chunks
3. Return those chunks as context

In [3]:
# Helper for Exercises 7 & 8: rebuild chunks + index with different chunk_size / chunk_overlap.
def rebuild_pipeline(chunk_size: int = 512, chunk_overlap: int = 128):
    """Re-chunk documents, re-embed, and rebuild FAISS index. Updates global all_chunks and index."""
    global all_chunks, index
    all_chunks = []
    for filename, content in documents:
        all_chunks.extend(chunk_text(content, filename, chunk_size=chunk_size, chunk_overlap=chunk_overlap))
    chunk_embeddings = embed_model.encode([c.text for c in all_chunks], show_progress_bar=True).astype("float32")
    faiss.normalize_L2(chunk_embeddings)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(chunk_embeddings)
    print(f"Rebuilt: {len(all_chunks)} chunks, chunk_size={chunk_size}, chunk_overlap={chunk_overlap}")

In [4]:
def retrieve(query: str, top_k: int = 5):
    """
    Retrieve the top-k most relevant chunks for a query.

    Returns: List of (chunk, similarity_score) tuples
    """
    # Embed the query
    query_embedding = embed_model.encode([query]).astype('float32')
    faiss.normalize_L2(query_embedding)

    # Search
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            results.append((all_chunks[idx], float(score)))

    return results

In [25]:
# Test retrieval
# ============================================
# TRY DIFFERENT QUERIES FOR YOUR CORPUS!
# ============================================
test_query = "What is the procedure for engine maintenance?"  # ← Modify this!

if index.ntotal > 0:
    results = retrieve(test_query, top_k=5)

    print(f"Query: {test_query}\n")
    print("Top 5 retrieved chunks:")
    for i, (chunk, score) in enumerate(results, 1):
        print(f"\n[{i}] Score: {score:.4f} | Source: {chunk.source_file}")
        print(f"    {chunk.text[:200]}...")
else:
    print("Index is empty - please load, chunk, and embed documents first.")

Query: What is the procedure for engine maintenance?

Top 5 retrieved chunks:

[1] Score: 0.7092 | Source: ATA_71.txt
    g, removal and installation
        for engine overhaul and engine replacement procedures.
2.   Maintenance Precautions
     A. The following maintenance precautions should be followed to prolong
    ...

[2] Score: 0.6438 | Source: ATA_71.pdf
    2. 
Maintenance Precautions 
A. 
The following maintenance precautions should be followed to prolong 
the life of the engine. Maintenance personnel should read and tho-
roughly understand these precau...

[3] Score: 0.6236 | Source: ATA_71.txt
    Dec 2/77
                                      ~
                             GAlEB LEARJET CORPORATION

                ■ai■ll!■a■II!· ■a■■al
                  GENERAL - MAINTENANCE PRACTICES

1.   ...

[4] Score: 0.5977 | Source: ATA_71.pdf
    haul Manual 
SEI-136 
CJ610 Turbojet Engine Illustrated Parts Catalog 
SEI-137 
EFFECTIVITY: ALL 
71-00-00 
Page 1 
Dec 2/77 


[Page 

---
## Stage 6: Generation (LLM)

Now we load a local LLM to generate answers from the retrieved context.

**Recommended models:**
- `Qwen/Qwen2.5-1.5B-Instruct` - Best instruction following at this size
- `Qwen/Qwen2.5-3B-Instruct` - Even better if you have 8GB+ VRAM
- `meta-llama/Llama-3.2-1B-Instruct` - Alternative, slightly weaker

**Device handling:**
- CUDA: Uses `device_map="auto"` and float16
- MPS: Loads to CPU first, then moves to MPS with float32
- CPU: Uses float32 (slower but works)

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# ============================================
# CHOOSE YOUR MODEL
# ============================================
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # Or try "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading LLM: {LLM_MODEL}")
print(f"Device: {DEVICE}, Dtype: {DTYPE}")
print("This may take a few minutes on first run...\n")

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

# Load with appropriate settings for each device type
if DEVICE == 'cuda':
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        device_map="auto",
        torch_dtype=DTYPE,
        trust_remote_code=True
    )
    print("Model loaded on CUDA")

elif DEVICE == 'mps':
    # For MPS, load to CPU first, then move to MPS
    # (device_map="auto" doesn't work well with MPS)
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        torch_dtype=DTYPE,
        trust_remote_code=True
    )
    model = model.to(DEVICE)
    print("Model loaded on MPS (Apple Silicon)")

else:
    # CPU
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        torch_dtype=DTYPE,
        trust_remote_code=True
    )
    print("Model loaded on CPU (this will be slow)")

Loading LLM: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda, Dtype: torch.float16
This may take a few minutes on first run...



config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on CUDA


In [7]:
def generate_response(prompt: str, max_new_tokens: int = 512, temperature: float = 0.3) -> str:
    """
    Generate a response from the LLM.

    Lower temperature = more focused/deterministic
    Higher temperature = more creative/random
    """
    inputs = tokenizer(prompt, return_tensors="pt")

    # Move inputs to the correct device
    if DEVICE == 'cuda':
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    else:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True if temperature > 0 else False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the new tokens
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()

---
## Stage 7: The Complete RAG Pipeline

Now we put it all together. The **prompt template** is critical - it must instruct the model to use the retrieved context.

In [57]:
# The RAG prompt template
PROMPT_TEMPLATE = """You are a helpful assistant that answers questions based on the provided context.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer the question based ONLY on the information in the context above
- If the context doesn't contain enough information to answer, say so
- Quote relevant parts of the context to support your answer
- Be concise and direct

ANSWER:"""


def direct_query(question: str, max_new_tokens: int = 512) -> str:
    """Ask the LLM directly with no retrieved context (for RAG vs no-RAG comparison)."""
    prompt = f"""Answer this question:
{question}

Answer:"""
    return generate_response(prompt, max_new_tokens=max_new_tokens)

def rag_query(question: str, top_k: int = 5, show_context: bool = False, prompt_template: str = None, thresh: float | None = None) -> str:
    """The complete RAG pipeline. prompt_template: custom template for Exercise 10."""
    # Step 1: Retrieve
    results = retrieve(question, top_k)

    if thresh:
      results = [result for result in results if result[1] > thresh]

    # Format context
    context_parts = []
    for chunk, score in results:
        context_parts.append(f"[Source: {chunk.source_file}, Relevance: {score:.3f}]\n{chunk.text}")
    context = "\n\n---\n\n".join(context_parts)

    if show_context:
        print("=" * 60)
        print("RETRIEVED CONTEXT:")
        print("=" * 60)
        print(context)
        print("=" * 60 + "\n")

    # Step 2: Build prompt (use custom template if provided)
    template = prompt_template if prompt_template is not None else PROMPT_TEMPLATE
    prompt = template.format(context=context, question=question)

    # Step 3: Generate
    answer = generate_response(prompt)

    return answer

In [26]:
# ============================================
# TEST YOUR RAG PIPELINE!
# ============================================

question = "What maintenance is required for the engine?"  # ← Modify for your corpus!

if index.ntotal > 0:
    print(f"Question: {question}\n")
    print("Generating answer...\n")

    answer = rag_query(question, top_k=5, show_context=True)

    print("ANSWER:")
    print(answer)
else:
    print("Pipeline not ready - please complete all previous stages first.")

Question: What maintenance is required for the engine?

Generating answer...

RETRIEVED CONTEXT:
[Source: ATA_71.txt, Relevance: 0.673]
g, removal and installation
        for engine overhaul and engine replacement procedures.
2.   Maintenance Precautions
     A. The following maintenance precautions should be followed to prolong
         the life of the engine. Maintenance personnel should read and tho-
         roughly understand these precautions.
         (1) Use.extreme care to prevent dirt, hardware, tools or other
              foreign material from entering the engine.
         (2) Handle.

---

[Source: ATA_71.pdf, Relevance: 0.667]
2. 
Maintenance Precautions 
A. 
The following maintenance precautions should be followed to prolong 
the life of the engine. Maintenance personnel should read and tho-
roughly understand these precautions. 
(1) Use.extreme care to prevent dirt, hardware, tools or other 
foreign material from entering the engine. 
(2) Handle. fuel and oil lines car

---
## Experiments: Understanding RAG Behavior

Now that you have a working pipeline, try these experiments to understand how each component affects the results.

In [ ]:
# EXPERIMENT 1: Compare WITH vs WITHOUT RAG
# ==========================================

question = "What are the specifications for the landing gear?"  # ← Use a corpus-specific question!

if index.ntotal > 0:
    # WITHOUT RAG - just ask the model directly
    direct_prompt = f"""Answer this question:
{question}

Answer:"""

    print("WITHOUT RAG (model's own knowledge):")
    print("-" * 40)
    direct_answer = generate_response(direct_prompt)
    print(direct_answer)

    print("\n" + "=" * 60 + "\n")

    # WITH RAG
    print("WITH RAG (using retrieved context):")
    print("-" * 40)
    rag_answer = rag_query(question, top_k=5)
    print(rag_answer)
else:
    print("Please complete the pipeline setup first.")

WITHOUT RAG (model's own knowledge):
----------------------------------------
The landing gear of an aircraft is a set of mechanisms that allow it to take off and land on an airport runway. It consists of two main parts - the wheels and the tailwheel (or tailskid). The wheels are located at the front of the aircraft, while the tailwheel or tailskid is located at the rear.
The landing gear can be either fixed or retractable. Fixed landing gears do not fold up when the plane is parked, but they provide better stability during takeoff and landing. Retractable landing gears, on the other hand, can be folded down into the fuselage when the plane is parked, making them easier to store and transport.
In terms of specifications, the size and weight of the landing gear depend on several factors such as the type of aircraft, its intended use, and the specific requirements of the airport where it will operate. For example, larger aircraft may require more powerful engines and heavier landing gear

In [ ]:
# EXPERIMENT 2: Effect of top_k
# ==========================================

question = "What safety procedures are required?"  # ← Use a corpus-specific question!

if index.ntotal > 0:
    for k in [1, 3, 5, 10]:
        print(f"\n{'='*60}")
        print(f"TOP_K = {k}")
        print(f"{'='*60}")
        answer = rag_query(question, top_k=k)
        print(answer[:500] + "..." if len(answer) > 500 else answer)
else:
    print("Please complete the pipeline setup first.")


TOP_K = 1
According to the given text, the safety precautions for rol panel maintenance include:

1. Pulling and tagging the appropriate system circuit breaker.
2. Ensuring that the Battery and Stall Warning Switches are turned off before lowering any instrument panel.
3. Using protective measures to shield the instrument face from damage during removal or installation processes. 

These steps aim to ensure the safety of personnel involved in rolling panel maintenance operations. The text specifies these ...

TOP_K = 3
According to the given context, the safety procedures required include:

1. Pulling and tagging an appropriate system circuit breaker.
2. Ensuring that the Battery and Stall Warning Switches are turned off before lowering any instrument panel.
3. Protecting the instrument face using suitable means. 

The context also mentions that these safety precautions are important for personnel who maintain the panels. It is recommended that they become familiar with these steps to

In [ ]:
# EXPERIMENT 3: Question the corpus CAN'T answer
# ==========================================
# Does the model admit it doesn't know, or hallucinate?

unanswerable_question = "What is the CEO's favorite color?"

if index.ntotal > 0:
    print(f"Question: {unanswerable_question}\n")
    answer = rag_query(unanswerable_question, top_k=5, show_context=True)
    print(f"\nAnswer: {answer}")
else:
    print("Please complete the pipeline setup first.")

Question: What is the CEO's favorite color?

RETRIEVED CONTEXT:
[Source: ATA_21.txt, Relevance: 0.439]
Orange


                                                                                                                                             , Brown
                                                                                                                                                                                   ..

---

[Source: CREC-2026-01-09.pdf, Relevance: 0.437]
R 
(Ms. ANSARI asked and was given 
permission to address the House for 1 
minute and to revise and extend her re-
marks.) 
Ms. ANSARI. Mr. Speaker, I rise to 
recognize the incredible accomplish-
ments of one of my district’s most tal-
ented public servants and the amazing 
organization that she helms. 
Monique Lopez was recently ap-
pointed as CEO of UMOM New Day 
Center.

---

[Source: ATA_21.txt, Relevance: 0.416]
fl                                                                               

---
## Save/Load Your Index

For large corpora, you don't want to re-embed every time. Here's how to persist the index.

In [12]:
import pickle

def save_index(filepath: str):
    """Save FAISS index and chunks to disk."""
    faiss.write_index(index, f"{filepath}.faiss")
    with open(f"{filepath}.chunks", 'wb') as f:
        pickle.dump(all_chunks, f)
    print(f"✓ Saved index to {filepath}.faiss")
    print(f"✓ Saved chunks to {filepath}.chunks")

def load_saved_index(filepath: str):
    """Load FAISS index and chunks from disk."""
    global index, all_chunks
    index = faiss.read_index(f"{filepath}.faiss")
    with open(f"{filepath}.chunks", 'rb') as f:
        all_chunks = pickle.load(f)
    print(f"✓ Loaded index with {index.ntotal} vectors")

# Save your index
try:
  if index.ntotal > 0:
      save_index("congressionalRecord_rag_index")
  else:
      print("No index to save.")
except NameError:
      print("No index to save.")

# Later, to load:
# load_saved_index("my_rag_index")

No index to save.


In [24]:
# Loading an index in order to not have to repeat previous loading cells.

load_saved_index("full_combined_corpora_rag_index")

✓ Loaded index with 139815 vectors


---
## Next Steps

You've built a complete RAG pipeline from scratch! In the next class, we'll:

1. **Improve retrieval** with query rewriting and hybrid search
2. **Rebuild with LangChain** to see how frameworks abstract these steps
3. **Evaluate systematically** with test questions and metrics

### Exercises to try:
- Vary chunk size (256, 512, 1024) and measure retrieval quality
- Try a different embedding model (`BAAI/bge-small-en-v1.5`)
- Try a larger LLM (`Qwen/Qwen2.5-3B-Instruct`) and compare answer quality
- Ask questions that require combining information from multiple chunks

---
## Appendix: Device Information

Run this cell to see detailed information about your compute environment.

In [ ]:
def print_device_info():
    """Print detailed information about available compute devices."""
    print("=" * 60)
    print("DEVICE INFORMATION")
    print("=" * 60)

    print(f"\nEnvironment: {ENVIRONMENT}")
    print(f"PyTorch version: {torch.__version__}")

    # CUDA
    print(f"\nCUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  Device: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    # MPS
    print(f"\nMPS available: {torch.backends.mps.is_available()}")
    print(f"MPS built: {torch.backends.mps.is_built()}")

    # Current selection
    print(f"\n→ Selected device: {DEVICE}")
    print(f"→ Selected dtype: {DTYPE}")
    print("=" * 60)

print_device_info()

DEVICE INFORMATION

Environment: colab
PyTorch version: 2.10.0+cu128

CUDA available: True
  Device: NVIDIA H100 80GB HBM3
  Memory: 85.0 GB

MPS available: False
MPS built: False

→ Selected device: cuda
→ Selected dtype: torch.float16


# Exercise 1: Open Model RAG vs. No RAG Comparison

In [ ]:
# Exercise 1: Open Model RAG vs. No RAG Comparison
import os

CORPORA_FILES = [
    "congressionalRecord_rag_index.faiss",
    "full_combined_corpora_rag_index.faiss",
    "modelTford_rag_index.faiss"
]

ex_1_query_set = [QUERIES_MODEL_T, QUERIES_CR]

print(f"{'='*80}")
print(f"{'='*80}")
print(f"WITHOUT RAG")
print(f"{'='*80}")
print(f"{'='*80}")

for query_set in ex_1_query_set:
    for query in query_set:
        print(f"\n{'-'*60}")
        print(f"QUESTION: {query}")
        print(f"{'-'*60}")

        print("\n>> RESPONSE:")
        print("-" * 20)
        print(direct_query(query))


print("\n")
print(f"{'='*80}")
print(f"{'='*80}")
print(f"WITH RAG")
print(f"{'='*80}")
print(f"{'='*80}")
print("\n")

for corpus_file in CORPORA_FILES:
    print(f"\n{'='*80}")
    print(f"CORPUS: {corpus_file}")
    print(f"{'='*80}")

    # Helper: strip extension because load_saved_index adds it back
    base_name = corpus_file.replace('.faiss', '')

    # Skip if file doesn't exist to prevent crashing
    if not os.path.exists(f"{base_name}.faiss"):
        continue

    load_saved_index(base_name)

    for query_set in ex_1_query_set:
        for query in query_set:
            print(f"\n{'-'*60}")
            print(f"QUESTION: {query}")
            print(f"{'-'*60}")
            print("\n>> RESPONSE (RAG):")
            print("-" * 20)
            print(rag_query(query))
            print("\n")

WITHOUT RAG

------------------------------------------------------------
QUESTION: How do I adjust the carburetor on a Model T?
------------------------------------------------------------

>> RESPONSE:
--------------------
To adjust the carburetor on a Model T, you need to follow these steps:

1. Locate and remove the air cleaner cover.
2. Remove the fuel line from the carburetor.
3. Use a screwdriver or wrench to loosen the needle valve adjustment bolt.
4. Turn the needle valve clockwise until it stops, then tighten it back down slightly.
5. Check if the engine runs smoothly by turning the ignition switch off and on several times.

Note: Be careful not to over-tighten the needle valve as this can damage the carburetor. Also, make sure that the carburetor is clean and free of debris before making any adjustments. If the problem persists after adjusting the carburetor, consider taking your car to a professional mechanic for further diagnosis and repair. Answer this question: How do I 

## RAG vs. no-RAG observations

### Model T manual queries
- **No RAG:** Frequently invents procedures and specs. Specs vary across runs -> low reliability.
- **With RAG:** Often pulls manual-adjacent instructions (e.g., band adjustment screw/lock nut; carburetor rod install). Still inconsistent on numeric specs (spark plug gap differs across indexes). Oil question not answered well (mostly “add new oil” / “approved oils”).

### Congressional Record queries
- **No RAG:** Makes up detailed claims about specific people/events.
- **With RAG:** Usually more grounded for “who said what” and bill purpose.

### Mixed (Model T + CR in one index)
- Benign when retrieval lands in the right corpus, but increases risk of wrong-corpus / wrong-passage hits.

# Exercise 2: Open Model + RAG vs. Large Model Comparison

In [ ]:
# Exercise 2: Frontier-ish Model (4o-mini) vs Open Model + RAG
# Install openai dependency
try:
    ip = get_ipython()
    ip.run_line_magic('pip', 'install -q openai')
except NameError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openai'])


In [ ]:
from openai import OpenAI
from google.colab import userdata

test_queries_flat = []
for query_set in ex_1_query_set:
    for query in query_set:
      test_queries_flat.append(query)

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
responses = [(question, client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[{"role":"user","content":f"""Answer this question:
  {question}

             Answer:"""}],
  max_completion_tokens=256,
).choices[0].message.content) for question in test_queries_flat]



[('How do I adjust the carburetor on a Model T?', 'Adjusting the carburetor on a Model T Ford involves a few key steps. The Model T typically uses a simple carburetor design, such as the NH or the earlier versions. Here’s a general guide on how to adjust it:\n\n1. **Warm Up the Engine**: Start the engine and let it warm up to operating temperature. This ensures that the adjustments you make are effective when the engine is running as it normally would.\n\n2. **Locate the Carburetor**: The carburetor is situated on the side of the engine. Familiarize yourself with its parts, including the float adjustment screw, the main adjustment needle, and the choke.\n\n3. **Check the Fuel Level**: First, make sure that the fuel tank is full and that fuel is reaching the carburetor. Inspect for any fuel flow issues.\n\n4. **Adjust the Main Needle Valve**: \n   - The main needle controls the fuel flow into the carburetor. Start by turning it in gently until it seats (do not overtighten) and then back

In [ ]:
for response in responses:
  print(f"Question: {response[0]}")
  print(f"\n>> Answer: {response[1]}\n")
  print("*" * 40)

Question: How do I adjust the carburetor on a Model T?

>> Answer: Adjusting the carburetor on a Model T Ford involves a few key steps. The Model T typically uses a simple carburetor design, such as the NH or the earlier versions. Here’s a general guide on how to adjust it:

1. **Warm Up the Engine**: Start the engine and let it warm up to operating temperature. This ensures that the adjustments you make are effective when the engine is running as it normally would.

2. **Locate the Carburetor**: The carburetor is situated on the side of the engine. Familiarize yourself with its parts, including the float adjustment screw, the main adjustment needle, and the choke.

3. **Check the Fuel Level**: First, make sure that the fuel tank is full and that fuel is reaching the carburetor. Inspect for any fuel flow issues.

4. **Adjust the Main Needle Valve**: 
   - The main needle controls the fuel flow into the carburetor. Start by turning it in gently until it seats (do not overtighten) and th

## Exercise 2 Results: Open Model + RAG vs. Large Model Comparison
> Open Model + RAG comparison reuses results from exercise 2.

**Comparison: Qwen 2.5 1.5B (with RAG) vs. GPT-4o Mini (No Tools)**

* **Hallucination Control:** GPT-4o Mini is significantly better at avoiding "low-level" hallucinations (like making up mechanical steps for a carburetor it doesn't know) but fails fundamentally on temporal data. It admitted ignorance for the 2026 Congressional Record queries, whereas Qwen 2.5 1.5B without RAG confidently invented a fake 2026 hearing for Mr. Flood.
* **Performance on Corpora:**
    * **Model T Ford:** GPT-4o Mini appeared to have performed well here. Because the Model T was designed in 1908, it is heavily represented in the pre-training data. It provided spark plug gaps (0.025") and oil recommendations without RAG.
    * **Congressional Record (Jan 2026):** GPT-4o Mini failed entirely, citing its October 2023 knowledge cutoff. In contrast, the small Qwen model with RAG was able to correctly identify Mr. Flood's remarks because the information was provided in the context window.
* **Conclusion:** A "Large" model is only better if the information is historical. For private or future-dated data (like your 2026 corpora), a small model with RAG is strictly superior to a frontier model without it.

# Exercise 9: Retrieval Score Analysis

In [50]:
# Expercise 9 Retrieval Score Analysis

corpora_rag_questions = [
"What is the main purpose of Regulation (EU) 2024/1689?",
"Which Treaty articles are cited as the legal basis for the EU AI Act?",
"What kinds of public interests and fundamental values does the EU AI Act say it is protecting?",
"In the Congressional Record for January 2, 2026, who was appointed Speaker pro tempore?",
"What bill is returned in the January 2, 2026 veto message, and what project is it tied to?",
"Across the January 2026 House entries, what themes stand out in the opening prayers?",
"In Learjet ATA Chapter 71, what engines power the aircraft and where are they installed?",
"According to Learjet Chapter 71, what systems or functions use compressor discharge (bleed) air?",
"According to the newer Ford service manual, what floor-space dimensions are given for the ideal shop layout?",
"In the newer Ford service manual, what does Chapter XLII cover?"
]

from statistics import mean, stdev
def analyze_retrieval_scores(question: str, top_k: int = 10, thresh: float | None = None):
    """Retrieve top-k chunks and compute score summary stats."""
    results = retrieve(question, top_k=top_k)
    if thresh:
        results = [result for result in results if result[1] > thresh]
    if not results:
        return {
            "question": question,
            "top_k": top_k,
            "scores": [],
            "mean_score": None,
            "std_dev_score": None,
            "chunks": [],
        }
    scores = [score for _, score in results]
    mean_score = mean(scores)
    std_dev_score = stdev(scores) if len(scores) > 1 else 0.0
    chunk_rows = []
    for rank, (chunk, score) in enumerate(results, start=1):
        chunk_rows.append(
            {
                "rank": rank,
                "score": float(score),
                "source_file": chunk.source_file,
                "chunk_index": chunk.chunk_index,
                "start_char": chunk.start_char,
                "end_char": chunk.end_char,
                "preview": chunk.text[:60].replace("\n", " "),
            }
        )
    return {
        "question": question,
        "top_k": min(top_k, len(results)),
        "scores": scores,
        "mean_score": mean_score,
        "std_dev_score": std_dev_score,
        "chunks": chunk_rows,
        "top_two_gap": scores[0] - scores[1] if len(scores) > 1 else None,
    }

for question in corpora_rag_questions:
  print(f"\n{'-'*60}")
  analysis = analyze_retrieval_scores(question, top_k=10)
  print(f"Question: {analysis['question']}")
  if analysis["scores"]:
      print(f"Scores: {[round(s, 4) for s in analysis['scores']]}")
      print(f"Sources used: {set([chunk['source_file'] for chunk in analysis['chunks']])}")
      print(f"Mean similarity: {analysis['mean_score']:.4f}")
      print(f"Std deviation: {analysis['std_dev_score']:.4f}")
      if analysis["top_two_gap"]:
        print(f"Gap between #1 and #2: {analysis['top_two_gap']:.4f} ({analysis['top_two_gap']/analysis['std_dev_score']:.4f} standard devations)")

#      print("\nTop chunks:")
#      for row in analysis["chunks"]:
#          print(
#              f"[{row['rank']}] score={row['score']:.4f} | {row['source_file']}"
#              f" | chunk={row['chunk_index']} ({row['start_char']}:{row['end_char']})"
#          )
#          print(f"    {row['preview']}...")
  else:
      print("No retrieval results. Ensure your index is built and non-empty.")




------------------------------------------------------------
Question: What is the main purpose of Regulation (EU) 2024/1689?
Scores: [0.7166, 0.6875, 0.6873, 0.6603, 0.6571, 0.6568, 0.6462, 0.6453, 0.6452, 0.6431]
Sources used: {'EU_AI_Act.pdf', 'EU_AI_Act.txt'}
Mean similarity: 0.6646
Std deviation: 0.0246
Gap between #1 and #2: 0.0291 (1.1834 standard devations)

------------------------------------------------------------
Question: Which Treaty articles are cited as the legal basis for the EU AI Act?
Scores: [0.7174, 0.7171, 0.6866, 0.6824, 0.6711, 0.67, 0.664, 0.651, 0.6464, 0.6447]
Sources used: {'EU_AI_Act.pdf', 'EU_AI_Act.txt'}
Mean similarity: 0.6751
Std deviation: 0.0263
Gap between #1 and #2: 0.0003 (0.0113 standard devations)

------------------------------------------------------------
Question: What kinds of public interests and fundamental values does the EU AI Act say it is protecting?
Scores: [0.6853, 0.6837, 0.6807, 0.6718, 0.6513, 0.6488, 0.6481, 0.6428, 0.6416, 0.6

In [53]:
# Applying score thresholds for comparison.
for thresh in [0.5, 0.6, 0.65, 0.675, .7]:
  print(f"\n{'='*80}")
  print(f"THRESHOLD: {thresh}")
  print(f"{'='*80}")
  for question in corpora_rag_questions[-3:]:
    print(f"\n{'-'*60}")
    analysis = analyze_retrieval_scores(question, top_k=10, thresh=thresh)
    print(f"Question: {analysis['question']}")
    if analysis["scores"]:
        print(f"Scores: {[round(s, 4) for s in analysis['scores']]}")
        print(f"Sources used: {set([chunk['source_file'] for chunk in analysis['chunks']])}")
        print(f"Mean similarity: {analysis['mean_score']:.4f}")
        print(f"Std deviation: {analysis['std_dev_score']:.4f}")
        if analysis["top_two_gap"]:
          print(f"Gap between #1 and #2: {analysis['top_two_gap']:.4f} ({analysis['top_two_gap']/analysis['std_dev_score']:.4f} standard devations)")
        else:
          print("Gap between #1 and #2: N/A")
    else:
        print(">> No retrieval results. Ensure your index is built and non-empty, and that threshold isn't too high.")



THRESHOLD: 0.5

------------------------------------------------------------
Question: According to Learjet Chapter 71, what systems or functions use compressor discharge (bleed) air?
Scores: [0.6926, 0.6788, 0.6785, 0.6689, 0.6457, 0.6396, 0.6371, 0.6346, 0.6211, 0.6203]
Sources used: {'ATA_36.pdf', 'ATA_36.txt', 'ATA_21.txt', 'ATA_71.pdf', 'ATA_21.pdf'}
Mean similarity: 0.6517
Std deviation: 0.0259
Gap between #1 and #2: 0.0138 (0.5324 standard devations)

------------------------------------------------------------
Question: According to the newer Ford service manual, what floor-space dimensions are given for the ideal shop layout?
Scores: [0.6181, 0.5772, 0.5611, 0.5556, 0.5483, 0.547, 0.5388, 0.5379, 0.525, 0.5071]
Sources used: {'ModelTNew.txt', 'ModelTNew.pdf'}
Mean similarity: 0.5516
Std deviation: 0.0303
Gap between #1 and #2: 0.0409 (1.3510 standard devations)

------------------------------------------------------------
Question: In the newer Ford service manual, what does 

## Exercise 9 Results: Retrieval Score Analysis

**Analysis of Similarity Scores (Top 10 Chunks)**

* **Score Distribution Patterns:**
    * **Clear Winners:** Queries like "Regulation (EU) 2024/1689" and "Model T Carburetor" showed a clear gap (~0.03–0.04) between the #1 and #2 results, indicating a highly specific semantic match.
    * **Clustered/Ambiguous:** Themes like "opening prayers" in the House resulted in very low, tightly clustered scores (around 0.48–0.50). This suggests the retriever found "vaguely religious" text across many documents but no "perfect" answer.
* **Threshold Recommendation:** Based on the logs, a **threshold of 0.60 to 0.65** is the sweet spot.
    * Above 0.65, results were almost always highly relevant.
    * Below 0.55, the pipeline started pulling "noise" (e.g., pulling Learjet engine data for a Model T question).
* **Effect of threshold on results:**  Using too low of a threshold resulted in unrelated sources being pulled in, which could negatively impact generation depending on how the question was prompted.

# Exercise 10: Prompt Template Variations

In [60]:
# Printing helper for Ex 10 and 11

def _clean_preview(text: str, max_chars: int = 280) -> str:
    flat = " ".join(text.strip().split())
    if len(flat) <= max_chars:
        return flat
    return flat[: max_chars - 3] + "..."

In [54]:
# Exercise 10: Prompt Template Variations
PROMPT_VARIANTS = {
    "minimal": """CONTEXT:
{context}

QUESTION: {question}

ANSWER:""",

    "strict_grounding": """You are a helpful assistant.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer ONLY based on the context above.
- If the answer is not in the context, say so.

ANSWER:""",

    "encouraging_citation": """You are a helpful assistant.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer based on the context above.
- Quote exact passages that support your answer.

ANSWER:""",

    "permissive": """You are a helpful assistant.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Use the context to help answer, but you may also use your knowledge.

ANSWER:""",

    "structured_output": """You are a helpful assistant.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- First list relevant facts from the context.
- Then synthesize your answer.

ANSWER:""",
}

EX10_QUERIES = [
    "What maintenance is required for the engine?",
    "What are the key safety warnings?",
    "What tools are required for a tune-up?",
    "How do I adjust the carburetor?",
    "What should I do monthly to maintain the system?",
]

EX10_TOP_K = 5
EX10_THRESH = None

print(f"Prompt variants: {list(PROMPT_VARIANTS.keys())}")
print(f"Queries: {len(EX10_QUERIES)} | top_k={EX10_TOP_K}")

Prompt variants: ['minimal', 'strict_grounding', 'encouraging_citation', 'permissive', 'structured_output']
Queries: 5 | top_k=5


In [59]:
def run_prompt_variations(queries, prompt_variants, top_k=5, thresh=None):
    rows = []

    for variant_name, template in prompt_variants.items():
        for query_idx, question in enumerate(queries, start=1):
            retrieved = retrieve(question, top_k=top_k)
            scores = [float(score) for _, score in retrieved]
            sources = sorted({chunk.source_file for chunk, _ in retrieved})

            answer = rag_query(
                question,
                top_k=top_k,
                prompt_template=template,
                thresh=thresh,
                show_context=False,
            )

            rows.append(
                {
                    "variant": variant_name,
                    "query_index": query_idx,
                    "question": question,
                    "top_k": top_k,
                    "retrieved_chunks": len(retrieved),
                    "unique_sources": len(sources),
                    "sources": sources,
                    "mean_similarity": (mean(scores) if scores else None),
                    "answer": answer,
                }
            )

    return rows


def print_ex10_readable(results, queries, prompt_order):
    print("\n" + "=" * 88)
    print("EXERCISE 10 RESULTS")
    print("=" * 88)

    # Detailed comparison by query
    print("\nDetailed answer comparison (by query)")
    print("-" * 88)
    for q_idx, question in enumerate(queries, start=1):
        print("\n" + "#" * 88)
        print(f"Q{q_idx}: {question}")
        print("#" * 88)
        for variant in prompt_order:
            row = next(
                r for r in results
                if r["variant"] == variant and r["query_index"] == q_idx
            )
            print(
                f"\n[{variant}] chunks={row['retrieved_chunks']}, "
                f"unique_sources={row['unique_sources']}, "
                f"mean_sim={row['mean_similarity']:.4f}" if row["mean_similarity"] is not None
                else f"\n[{variant}] chunks={row['retrieved_chunks']}, unique_sources={row['unique_sources']}, mean_sim=None"
            )
            print(f"Sources: {', '.join(row['sources'][:4])}" + (" ..." if len(row["sources"]) > 4 else ""))
            print(f"Answer: {_clean_preview(row['answer'], max_chars=420)}")


ex10_results = run_prompt_variations(
    EX10_QUERIES,
    PROMPT_VARIANTS,
    top_k=EX10_TOP_K,
    thresh=EX10_THRESH,
)

print_ex10_readable(
    ex10_results,
    EX10_QUERIES,
    prompt_order=list(PROMPT_VARIANTS.keys()),
)


EXERCISE 10 RESULTS

Detailed answer comparison (by query)
----------------------------------------------------------------------------------------

########################################################################################
Q1: What maintenance is required for the engine?
########################################################################################

[minimal] chunks=5, unique_sources=3, mean_sim=0.6270
Sources: ATA_27.txt, ATA_71.pdf, ATA_71.txt
Answer: Removal and installation for engine overhaul and engine replacement procedures. This answer is derived directly from the given text which states "Maintenance practices include servicing, removal and installation for engine overhaul and engine replacement procedures." This indicates that both removal and installation are part of the maintenance process for engines. However, it does not specify what specific types o...

[strict_grounding] chunks=5, unique_sources=3, mean_sim=0.6270
Sources: ATA_27.txt, ATA_71.pdf

## Exercise 10 Results: Prompt Template Variations

**Overall Conclusion**

The results were pretty consistent across prompt templates in most circumstances, however there were a few instances where the various templates resulted in meaningful differences in output.

**Evaluation of System Prompts on Generation Quality**

- Accuracy: strict_grounding was the most accurate for preventing hallucinations. It forced the model to admit when information was missing rather than guessing.

- Usefulness: structured_output produced the most useful results. The explicit fact-listing phase acted made the final synthesis appear more systematic and grounded.

- The Trade-off: There is a clear tension between strict grounding and helpfulness. Strict grounding ensures accuracy but results in a "stiff" user experience (even resulting in one answer being totally imcomplete), whereas more permissive prompts feel more natural but introduces data not present in the corpora.

# Exercise 11: Cross-Document Synthesis
> We use the ``structured_output`` prompt variant for this exercise.

In [62]:
# Exercise 11: Cross-Document Synthesis
EX11_QUERIES = [
    "What are all the maintenance tasks I should do monthly?",
    "Compare the procedures for adjusting X versus adjusting Y.",
    "What tools are required for a complete tune-up?",
    "Summarize all safety warnings in the manual.",
    "Which maintenance tasks are safety-critical and why?",
]

EX11_TOP_K_VALUES = [3, 5, 10]
EX11_PROMPT_TEMPLATE = PROMPT_VARIANTS["structured_output"]
EX11_THRESH = None

print(f"Synthesis queries: {len(EX11_QUERIES)}")
print(f"Top-k values: {EX11_TOP_K_VALUES}")

Synthesis queries: 5
Top-k values: [3, 5, 10]


In [66]:
def run_ex11_cross_document(queries, top_k_values, prompt_template, thresh=None):
    rows = []

    for top_k in top_k_values:
        for query_idx, question in enumerate(queries, start=1):
            retrieved = retrieve(question, top_k=top_k)
            sources = sorted({chunk.source_file for chunk, _ in retrieved})

            answer = rag_query(
                question,
                top_k=top_k,
                prompt_template=prompt_template,
                thresh=thresh,
                show_context=False,
            )

            rows.append(
                {
                    "top_k": top_k,
                    "query_index": query_idx,
                    "question": question,
                    "retrieved_chunks": len(retrieved),
                    "unique_sources": len(sources),
                    "multi_source": len(sources) > 1,
                    "sources": sources,
                    "answer": answer,
                }
            )

    return rows


def print_ex11_readable(results, queries, top_k_values):
    print("\n" + "=" * 88)
    print("EXERCISE 11 RESULTS")
    print("=" * 88)

    # Compact summary by top_k
    print("\nTop-k retrieval summary")
    print("-" * 88)
    for k in top_k_values:
        rows = [r for r in results if r["top_k"] == k]
        avg_unique_sources = mean([r["unique_sources"] for r in rows]) if rows else 0.0
        pct_multi = 100.0 * mean([1.0 if r["multi_source"] else 0.0 for r in rows]) if rows else 0.0
        print(
            f"k={k:<2d} | avg unique sources: {avg_unique_sources:.2f} | "
            f"multi-source retrieval: {pct_multi:.1f}%"
        )

    # Detailed by query, comparing k values
    print("\nDetailed synthesis comparison (by query, across k)")
    print("-" * 88)
    for q_idx, question in enumerate(queries, start=1):
        print("\n" + "#" * 88)
        print(f"Q{q_idx}: {question}")
        print("#" * 88)

        for k in top_k_values:
            row = next(
                r for r in results
                if r["query_index"] == q_idx and r["top_k"] == k
            )
            print(
                f"\n[k={k}] chunks={row['retrieved_chunks']}, "
                f"unique_sources={row['unique_sources']}, "
                f"multi_source={row['multi_source']}"
            )
            print(f"Sources: {', '.join(row['sources'][:10])}" + (" ..." if len(row["sources"]) > 10 else ""))
            print(f"Answer: {_clean_preview(row['answer'], max_chars=460)}")


ex11_results = run_ex11_cross_document(
    EX11_QUERIES,
    EX11_TOP_K_VALUES,
    prompt_template=EX11_PROMPT_TEMPLATE,
    thresh=EX11_THRESH,
)

print_ex11_readable(ex11_results, EX11_QUERIES, EX11_TOP_K_VALUES)


EXERCISE 11 RESULTS

Top-k retrieval summary
----------------------------------------------------------------------------------------
k=3  | avg unique sources: 3.00 | multi-source retrieval: 100.0%
k=5  | avg unique sources: 4.80 | multi-source retrieval: 100.0%
k=10 | avg unique sources: 8.80 | multi-source retrieval: 100.0%

Detailed synthesis comparison (by query, across k)
----------------------------------------------------------------------------------------

########################################################################################
Q1: What are all the maintenance tasks I should do monthly?
########################################################################################

[k=3] chunks=3, unique_sources=3, multi_source=True
Sources: ATA_39.txt, ATA_71.txt, ATA_77.txt
Answer: Monthly maintenance tasks include: 1. Electrical checks on all systems to ensure proper functioning. 2. Replacement of any faulty components as per the guidelines provided in the text. 

## Exercise 11 Results: Cross-Document Synthesis

**Analysis of Model Performance Across k=3, k=5, and k=10**

* **Synthesis Capability:** The model successfully aggregated lists (e.g., safety warnings) from multiple files but failed at complex comparisons. When asked to compare procedures, it often concluded they were identical because the retrieved text lacked specific labels for comparison.
* **Impact of top_k:** At k=3, the model occasionally "forced" an answer from limited data. At k=10, the increased context allowed the model to self-correct and admit when specific information (like a monthly schedule) was actually missing from the manual.
* **Data Gaps:** Synthesis did not compensate for missing information. If a task wasn't in the top-k results, the model couldn't "reason" its existence. Retrieval noise at k=10 was generally ignored during the final synthesis step.

**Documented Findings**

* **Success:** The model is a reliable list aggregator for broad summaries but struggles with multi-hop logical tasks.
* **Missing Info:** The model is strictly bound by retrieval; it cannot synthesize facts that were not captured in the top-k chunks.
* **Conflicts:** No major contradictions occurred; the model often defaulted to a "not mentioned" or identical response when faced with ambiguity.